# 🌊 Flood-AI: 01 - Multimodal Data Ingestion & Quality Validation
### End-to-End Ingestion across 20 Official Multi-Sensor Catalog Sources
This notebook loads, audits, and aligns heterogeneous remote sensing rasters, river streamgage telemetry, gridded climate reanalysis, demographic grids, and crowdsourced crisis reports.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Setup paths
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.dataset_registry import DatasetRegistry

print(f"✓ Project Root: {project_root}")
print(f"✓ Registered Datasets: {len(DatasetRegistry.load_catalog())}")

## 1. Load Satellite SAR (Sentinel-1) & Optical (Sentinel-2) Arrays
Loading calibrated dual-pol SAR backscatter ($\sigma^0_{VV}$ and $\sigma^0_{VH}$) and optical NDWI spectral indices from Sen1Floods11 and Sen2Floods.

In [ ]:
raw_dir = project_root / "data" / "raw"
vv_chips = np.load(raw_dir / "satellite" / "sen1floods11" / "s1_sar_vv_chips.npy")
vh_chips = np.load(raw_dir / "satellite" / "sen1floods11" / "s1_sar_vh_chips.npy")
masks = np.load(raw_dir / "satellite" / "sen1floods11" / "ground_truth_flood_masks.npy")
ndwi = np.load(raw_dir / "satellite" / "sen2floods" / "s2_ndwi_chips.npy")
sat_meta = pd.read_csv(raw_dir / "satellite" / "sen1floods11" / "metadata.csv")

print(f"✓ Sentinel-1 VV Chips Shape: {vv_chips.shape} (Range: {vv_chips.min():.1f} to {vv_chips.max():.1f} dB)")
print(f"✓ Sentinel-1 VH Chips Shape: {vh_chips.shape} (Range: {vh_chips.min():.1f} to {vh_chips.max():.1f} dB)")
print(f"✓ Ground Truth Masks Shape:  {masks.shape} (Mean water fraction: {masks.mean()*100:.2f}%)")
print(f"✓ Sentinel-2 NDWI Shape:     {ndwi.shape}")
sat_meta.head()

## 2. Ingest Hydrological Streamgages & Meteorological Telemetry
Loading 15-minute continuous river sensor records (USGS NWIS, BWDB, CWC) and hourly weather reanalysis (ERA5, GPM IMERG).

In [ ]:
df_gauge = pd.read_csv(raw_dir / "hydrology" / "usgs_bwdb" / "river_gauge_telemetry.csv")
df_weather = pd.read_csv(raw_dir / "weather" / "era5_gpm" / "hourly_weather_precipitation.csv")

print(f"✓ River Gauge Records: {len(df_gauge):,} entries across {df_gauge['site_name'].nunique()} stations")
print(f"✓ Weather Records:     {len(df_weather):,} hourly entries across {df_weather['city'].nunique()} global basins")
print("\n--- River Gauge Sample ---")
print(df_gauge[["timestamp", "site_name", "gauge_height_m", "discharge_cfs", "status"]].head(3))

## 3. Ingest Exposure, Structural Damage & Citizen Crisis Reports
Loading WorldPop 100m population density, DARPA xBD building damage classifications, and CrisisMMD/HumAID citizen eyewitness incident reports.

In [ ]:
df_xbd = pd.read_csv(raw_dir / "satellite" / "xbd" / "xbd_building_damage.csv")
df_pop = pd.read_csv(raw_dir / "exposure" / "worldpop" / "worldpop_density_grids.csv")
df_social = pd.read_csv(raw_dir / "social" / "crisismmd_humaid" / "crisis_social_reports.csv")
df_osm = pd.read_csv(raw_dir / "vector" / "osm" / "osm_shelters_infrastructure.csv")

print(f"✓ xBD Building Footprints: {len(df_xbd):,} records ({df_xbd['damage_level'].value_counts().to_dict()})")
print(f"✓ WorldPop Grid Cells:     {len(df_pop):,} population cells")
print(f"✓ CrisisMMD Incident Feed: {len(df_social):,} citizen reports ({df_social['humanitarian_category'].value_counts().to_dict()})")
print(f"✓ OSM Evacuation Shelters: {len(df_osm):,} verified high-ground facilities")

## 4. Export Ingestion Summary Manifest

In [ ]:
manifest = {
    "num_sar_chips": len(vv_chips),
    "num_gauge_records": len(df_gauge),
    "num_weather_hours": len(df_weather),
    "num_xbd_buildings": len(df_xbd),
    "num_crisis_reports": len(df_social),
    "status": "DATA_INGESTION_COMPLETE"
}
proc_dir = project_root / "data" / "processed"
proc_dir.mkdir(parents=True, exist_ok=True)
import json
with open(proc_dir / "ingestion_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("✅ 01 Data Ingestion Stage Complete! Manifest saved.")